# Import and file check

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'torch_xla','albumentations', '-q'])
subprocess.run(['pip', 'install', 'torch_xla', '-q'])

import os, math, copy, random
import numpy as np
import pandas as pd
import torch
import torchvision
import xml.etree.ElementTree as ET
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import functional as F
import torchvision.transforms as T
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FasterRCNN, FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import nms, box_iou

print("PyTorch    :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device     :", "GPU ✓" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    print("VRAM       :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:15]:
        print(os.path.join(dirname, filename))

PyTorch    : 2.10.0+cpu
Torchvision: 0.25.0+cpu
Device     : CPU
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/submission.csv
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/pitted_surface_gay7i73fhffn.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/patches_1zv1k18lbav2.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_rcfzhlihrdau.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_fvmm5t1evw7y.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_pcgal0tq7lw4.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/patches_i31v5waomc1h.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/inclusion_zcmc5571s37u.jpg
/kaggle/input/competitions/teep-internship-program-huce

# Configuration

In [2]:

DATASET_ROOT   = '/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset'
TRAIN_IMG_ROOT = DATASET_ROOT + '/train_images'
TRAIN_ANN_ROOT = DATASET_ROOT + '/train_annotations'
VAL_IMG_ROOT   = DATASET_ROOT + '/test'

# ── Core settings ─────────────────────────────────────────────
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES    = 7          # 6 defects + background
EPOCHS         = 50         # more epochs = more improvement
BATCH_SIZE     = 4
LEARNING_RATE  = 3e-4       # higher LR with warmup works better
WEIGHT_DECAY   = 1e-4
CONF_THRESHOLD = 0.35       # lower = more detections = better recall
NMS_THRESHOLD  = 0.45
USE_TTA        = True
INFER_ONLY     = False
IMG_SIZE       = 640        # resize all images to this for consistency
SEED           = 42

# ── Output ────────────────────────────────────────────────────
OUTPUT_DIR      = '/kaggle/working'
SUBMISSION_PATH = OUTPUT_DIR + '/submission.csv'
CHECKPOINT_PATH = OUTPUT_DIR + '/checkpoint_best.pth'
EMA_PATH        = OUTPUT_DIR + '/ema_model.pth'
VIZ_DIR         = OUTPUT_DIR + '/output_viz'
os.makedirs(VIZ_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Labels ────────────────────────────────────────────────────
LABEL_MAP = {
    'crazing':         1,
    'inclusion':       2,
    'patches':         3,
    'pitted_surface':  4,
    'rolled-in_scale': 5,
    'scratches':       6,
}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES   = ['bg'] + list(LABEL_MAP.keys())

# Confirm paths
for name, path in [('train_images', TRAIN_IMG_ROOT),
                   ('annotations',  TRAIN_ANN_ROOT),
                   ('test',         VAL_IMG_ROOT)]:
    ok = os.path.exists(path)
    n  = len(os.listdir(path)) if ok else 0
    print(f"{'OK' if ok else 'MISSING':6s}  {name:20s}  ({n} files)  {path}")

OK      train_images          (1440 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images
OK      annotations           (1440 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_annotations
OK      test                  (360 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/test


#  Augmentation (Albumentations — much stronger than torchvision)

In [3]:
def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),

        # Spatial
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2,
                           rotate_limit=15, p=0.5,
                           border_mode=0),

        # Photometric — critical for grayscale metal images
        A.RandomBrightnessContrast(brightness_limit=0.3,
                                   contrast_limit=0.3, p=0.6),
        A.GaussNoise(var_limit=(10, 50), p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.CLAHE(clip_limit=4.0, p=0.4),         # enhances defect edges
        A.Sharpen(alpha=(0.2, 0.5), p=0.3),

        # Cutout — forces model to detect partial defects
        A.CoarseDropout(max_holes=6, max_height=32,
                        max_width=32, fill_value=0, p=0.3),

        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        min_area=16,
        min_visibility=0.3,
    ))


def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        min_area=1,
        min_visibility=0.1,
    ))

print("Augmentations ready  (Albumentations)")

Augmentations ready  (Albumentations)


#  Dataset with class-balanced sampler

In [4]:
class NEUDETDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir    = img_dir
        self.ann_dir    = ann_dir
        self.transforms = transforms
        self.imgs = sorted([
            f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')
        ])
        # preload all annotations for sampler
        self.all_labels = []
        for fname in self.imgs:
            ann = os.path.join(ann_dir, fname.replace('.jpg', '.xml'))
            lbls = []
            for obj in ET.parse(ann).getroot().findall('object'):
                lbls.append(LABEL_MAP[obj.find('name').text.strip()])
            self.all_labels.append(lbls[0] if lbls else 0)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        image    = np.array(Image.open(
            os.path.join(self.img_dir, img_name)).convert('RGB'))
        ann_path = os.path.join(self.ann_dir, img_name.replace('.jpg', '.xml'))

        boxes, labels = [], []
        for obj in ET.parse(ann_path).getroot().findall('object'):
            lbl  = obj.find('name').text.strip()
            bbox = obj.find('bndbox')
            xmin = int(float(bbox.find('xmin').text))
            ymin = int(float(bbox.find('ymin').text))
            xmax = int(float(bbox.find('xmax').text))
            ymax = int(float(bbox.find('ymax').text))
            h, w = image.shape[:2]
            xmin, xmax = max(0, xmin), min(w, xmax)
            ymin, ymax = max(0, ymin), min(h, ymax)
            if xmax > xmin + 2 and ymax > ymin + 2:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(LABEL_MAP[lbl])

        if self.transforms and boxes:
            try:
                aug = self.transforms(
                    image=image,
                    bboxes=boxes,
                    class_labels=labels,
                )
                image  = aug['image']
                boxes  = list(aug['bboxes'])
                labels = list(aug['class_labels'])
            except Exception:
                image = torch.from_numpy(
                    image.transpose(2,0,1)).float() / 255.0
        elif self.transforms:
            aug    = self.transforms(image=image, bboxes=[], class_labels=[])
            image  = aug['image']
        else:
            image = torch.from_numpy(
                image.transpose(2,0,1)).float() / 255.0

        boxes  = (torch.as_tensor(boxes,  dtype=torch.float32)
                  if boxes else torch.zeros((0,4), dtype=torch.float32))
        labels = (torch.as_tensor(labels, dtype=torch.int64)
                  if labels else torch.zeros((0,), dtype=torch.int64))

        target = {
            'boxes':    boxes,
            'labels':   labels,
            'image_id': torch.tensor([idx]),
            'area':     ((boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0])
                         if len(boxes) else torch.zeros(0)),
            'iscrowd':  torch.zeros(len(labels), dtype=torch.int64),
        }
        return image, target

    def __len__(self):
        return len(self.imgs)


def get_balanced_sampler(dataset):
    """Over-sample rare classes so every class gets equal training exposure."""
    label_counts = np.bincount(dataset.all_labels, minlength=NUM_CLASSES)
    label_counts = np.maximum(label_counts, 1)
    weights = 1.0 / label_counts
    sample_weights = torch.tensor(
        [weights[l] for l in dataset.all_labels], dtype=torch.float)
    return WeightedRandomSampler(sample_weights, len(sample_weights))


def collate_fn(batch):
    return tuple(zip(*batch))


# Sanity check
_ds = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT)
_img, _tgt = _ds[0]
print(f"Train samples : {len(_ds)}")
print(f"Image shape   : {_img.shape if isinstance(_img, torch.Tensor) else np.array(_img).shape}")
print(f"Boxes         : {_tgt['boxes']}")
print(f"Labels        : {_tgt['labels']}")

# Class distribution
from collections import Counter
dist = Counter(_ds.all_labels)
print("\nClass distribution:")
for cls_id, count in sorted(dist.items()):
    name = REV_LABEL_MAP.get(cls_id, 'background')
    print(f"  {name:20s}: {count}")

Train samples : 1440
Image shape   : torch.Size([3, 200, 200])
Boxes         : tensor([[ 90., 117., 164., 159.],
        [  3.,  42.,  72.,  93.]])
Labels        : tensor([1, 1])

Class distribution:
  crazing             : 240
  inclusion           : 240
  patches             : 240
  pitted_surface      : 240
  rolled-in_scale     : 240
  scratches           : 240


# Model (with internet OFF fallback)

In [5]:
def get_model(num_classes=NUM_CLASSES):
    """
    Primary:  ResNet-101 FPN  (needs internet)
    Fallback: ResNet-50  FPN  (pretrained COCO, offline)

    Key changes vs baseline:
    - Smaller anchors (16px) for tiny defects
    - Wide aspect ratios for elongated scratches
    - More detections per image (100)
    - Lower score threshold for better recall
    """
    try:
        backbone = resnet_fpn_backbone(
            backbone_name='resnet101',
            weights='ResNet101_Weights.IMAGENET1K_V1',
            trainable_layers=4,
        )
        anchor_gen = AnchorGenerator(
            sizes=((16,), (32,), (64,), (128,), (256,)),
            aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5,
        )
        model = FasterRCNN(
            backbone,
            num_classes=num_classes,
            rpn_anchor_generator=anchor_gen,
            box_detections_per_img=200,
            rpn_nms_thresh=0.7,
            box_nms_thresh=NMS_THRESHOLD,
            box_score_thresh=CONF_THRESHOLD,
            # Larger RPN proposals = more candidate regions
            rpn_pre_nms_top_n_train=4000,
            rpn_post_nms_top_n_train=2000,
            rpn_pre_nms_top_n_test=2000,
            rpn_post_nms_top_n_test=1000,
        )
        print("Model: Faster R-CNN ResNet-101 FPN  (pretrained ImageNet)")

    except Exception as e:
        print(f"ResNet-101 failed ({e.__class__.__name__}), using ResNet-50 COCO weights...")
        model = fasterrcnn_resnet50_fpn(
            weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        )
        # Replace head for 7 classes
        in_feat = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)

        # Swap anchor generator even on ResNet-50
        anchor_gen = AnchorGenerator(
            sizes=((16,), (32,), (64,), (128,), (256,)),
            aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5,
        )
        model.rpn.anchor_generator = anchor_gen

        # Tune inference params
        model.rpn.nms_thresh               = 0.7
        model.roi_heads.nms_thresh         = NMS_THRESHOLD
        model.roi_heads.score_thresh       = CONF_THRESHOLD
        model.roi_heads.detections_per_img = 200

        print("Model: Faster R-CNN ResNet-50 FPN  (pretrained COCO)")

    return model.to(DEVICE)


model = get_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 182MB/s] 


Model: Faster R-CNN ResNet-101 FPN  (pretrained ImageNet)
Trainable parameters: 60,257,852


#  mAP evaluation helper

In [6]:
def compute_map(model, val_loader, iou_threshold=0.5, device=DEVICE):
    """
    Compute per-class AP and mAP on a validation set.
    Uses VOC-style 11-point interpolation.
    """
    model.eval()
    all_preds  = {i: [] for i in range(1, NUM_CLASSES)}
    all_gts    = {i: [] for i in range(1, NUM_CLASSES)}
    img_offset = 0

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Evaluating", leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for i, (out, tgt) in enumerate(zip(outputs, targets)):
                gt_boxes  = tgt['boxes'].cpu()
                gt_labels = tgt['labels'].cpu()
                pd_boxes  = out['boxes'].cpu()
                pd_scores = out['scores'].cpu()
                pd_labels = out['labels'].cpu()
                img_id    = img_offset + i

                for cls in range(1, NUM_CLASSES):
                    gt_mask = gt_labels == cls
                    pd_mask = pd_labels == cls
                    all_gts[cls].append({
                        'img_id': img_id,
                        'boxes':  gt_boxes[gt_mask],
                        'matched': torch.zeros(gt_mask.sum(), dtype=torch.bool),
                    })
                    scores = pd_scores[pd_mask]
                    boxes  = pd_boxes[pd_mask]
                    order  = scores.argsort(descending=True)
                    all_preds[cls].append({
                        'img_id': img_id,
                        'boxes':  boxes[order],
                        'scores': scores[order],
                    })
            img_offset += len(images)

    aps = {}
    for cls in range(1, NUM_CLASSES):
        preds_flat = []
        for p in all_preds[cls]:
            for b, s in zip(p['boxes'], p['scores']):
                preds_flat.append((p['img_id'], float(s), b))
        preds_flat.sort(key=lambda x: -x[1])

        gt_map = {p['img_id']: p for p in all_gts[cls]}
        n_gt   = sum(len(p['boxes']) for p in all_gts[cls])
        if n_gt == 0:
            aps[cls] = 0.0
            continue

        tp = torch.zeros(len(preds_flat))
        fp = torch.zeros(len(preds_flat))

        for idx, (img_id, score, box) in enumerate(preds_flat):
            gt_entry = gt_map.get(img_id, None)
            if gt_entry is None or len(gt_entry['boxes']) == 0:
                fp[idx] = 1; continue

            ious = box_iou(box.unsqueeze(0), gt_entry['boxes'])[0]
            best_iou, best_j = ious.max(0) if len(ious) else (torch.tensor(0.), torch.tensor(0))

            if best_iou >= iou_threshold and not gt_entry['matched'][best_j]:
                tp[idx] = 1
                gt_entry['matched'][best_j] = True
            else:
                fp[idx] = 1

        cum_tp = tp.cumsum(0)
        cum_fp = fp.cumsum(0)
        rec  = cum_tp / max(n_gt, 1)
        prec = cum_tp / (cum_tp + cum_fp + 1e-9)

        # 11-point interpolated AP
        ap = 0.0
        for t in torch.linspace(0, 1, 11):
            p_at_r = prec[rec >= t].max() if (rec >= t).any() else torch.tensor(0.)
            ap += p_at_r.item() / 11
        aps[cls] = ap

    map_score = np.mean(list(aps.values()))
    print(f"\nmAP@0.5 = {map_score:.4f}")
    for cls_id, ap in aps.items():
        print(f"  {REV_LABEL_MAP[cls_id]:20s}: AP = {ap:.4f}")
    return map_score, aps


print("mAP evaluator ready")

mAP evaluator ready


# Training loop

In [7]:
class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd  = model.state_dict()
        emsd = self.ema.state_dict()
        for k in emsd:
            emsd[k].mul_(self.decay).add_(msd[k], alpha=1-self.decay)
        self.ema.load_state_dict(emsd)


def warmup_cosine_schedule(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


print("EMA & scheduler ready")

EMA & scheduler ready


#  TTA predict

In [8]:
def train_model(model, train_loader, val_loader=None, epochs=EPOCHS):
    # Separate LR for backbone (lower) vs head (higher)
    backbone_params = []
    head_params     = []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'backbone' in name:
            backbone_params.append(p)
        else:
            head_params.append(p)

    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LEARNING_RATE * 0.1},
        {'params': head_params,     'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = warmup_cosine_schedule(
        optimizer, warmup_epochs=3, total_epochs=epochs,
        steps_per_epoch=len(train_loader)
    )
    ema       = ModelEMA(model, decay=0.9999)
    best_map  = 0.0
    history   = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        loss_components = {'cls': 0, 'box': 0, 'rpn_cls': 0, 'rpn_box': 0}
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:3d}/{epochs}")

        for images, targets in pbar:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss      = sum(loss_dict.values())

            # Track components for debugging
            for k, v in loss_dict.items():
                short = k.replace('loss_', '')
                if short in loss_components:
                    loss_components[short] += v.item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                backbone_params + head_params, max_norm=5.0)
            optimizer.step()
            scheduler.step()
            ema.update(model)

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.3f}",
                             lr=f"{scheduler.get_last_lr()[-1]:.1e}")

        avg_loss = epoch_loss / len(train_loader)
        row = {'epoch': epoch+1, 'loss': avg_loss,
               'lr': scheduler.get_last_lr()[-1]}

        # Evaluate mAP every 5 epochs
        map_score = 0.0
        if val_loader and (epoch+1) % 5 == 0:
            map_score, _ = compute_map(ema.ema, val_loader)
            model.train()
            row['map'] = map_score

        history.append(row)
        print(f"  loss={avg_loss:.4f}  "
              f"cls={loss_components['cls']/len(train_loader):.3f}  "
              f"box={loss_components['box']/len(train_loader):.3f}  "
              f"lr={scheduler.get_last_lr()[-1]:.2e}"
              + (f"  mAP={map_score:.4f}" if map_score else ""))

        # Save best by mAP if available, else by loss
        save = (map_score > best_map) if map_score else (avg_loss < getattr(train_model, '_best_loss', 1e9))
        if save:
            best_map = map_score if map_score else best_map
            train_model._best_loss = avg_loss
            torch.save({'model': model.state_dict(),
                        'epoch': epoch, 'best_map': best_map,
                        'best_loss': avg_loss}, CHECKPOINT_PATH)
            torch.save(ema.ema.state_dict(), EMA_PATH)
            print(f"  ✓ Saved best checkpoint")

    # Plot
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR + '/train_history.csv', index=False)

    fig, axes = plt.subplots(1, 2 if 'map' in hist_df.columns else 1, figsize=(14, 4))
    if not isinstance(axes, np.ndarray): axes = [axes]
    axes[0].plot(hist_df['epoch'], hist_df['loss'], marker='o', ms=2, linewidth=1.5)
    axes[0].set(xlabel='Epoch', ylabel='Loss', title='Training Loss')
    axes[0].grid(alpha=0.3)
    if 'map' in hist_df.columns:
        map_df = hist_df.dropna(subset=['map'])
        axes[1].plot(map_df['epoch'], map_df['map'], marker='s', ms=4,
                     color='green', linewidth=1.5)
        axes[1].set(xlabel='Epoch', ylabel='mAP@0.5', title='Validation mAP')
        axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + '/training_curves.png', dpi=120)
    plt.show()

    return ema.ema

#  Predict & export

In [9]:
@torch.no_grad()
def tta_predict(model, image_tensor):
    """3-view TTA: original + h-flip + v-flip, merged via NMS."""
    outputs = []

    # Original
    out = model(image_tensor)[0]
    outputs.append(out)

    # Horizontal flip
    img_hf  = F.hflip(image_tensor[0]).unsqueeze(0)
    out_hf  = model(img_hf)[0]
    w       = image_tensor.shape[-1]
    b_hf    = out_hf['boxes'].clone()
    b_hf[:, [0,2]] = w - b_hf[:, [2,0]]
    outputs.append({'boxes': b_hf, 'scores': out_hf['scores'],
                    'labels': out_hf['labels']})

    # Vertical flip
    img_vf  = F.vflip(image_tensor[0]).unsqueeze(0)
    out_vf  = model(img_vf)[0]
    h       = image_tensor.shape[-2]
    b_vf    = out_vf['boxes'].clone()
    b_vf[:, [1,3]] = h - b_vf[:, [3,1]]
    outputs.append({'boxes': b_vf, 'scores': out_vf['scores'],
                    'labels': out_vf['labels']})

    all_boxes  = torch.cat([o['boxes']  for o in outputs])
    all_scores = torch.cat([o['scores'] for o in outputs])
    all_labels = torch.cat([o['labels'] for o in outputs])

    if len(all_boxes) == 0:
        return outputs[0]

    # Per-class NMS so different classes don't suppress each other
    final_boxes, final_scores, final_labels = [], [], []
    for cls in all_labels.unique():
        mask   = all_labels == cls
        keep   = nms(all_boxes[mask], all_scores[mask], iou_threshold=0.45)
        final_boxes.append(all_boxes[mask][keep])
        final_scores.append(all_scores[mask][keep])
        final_labels.append(all_labels[mask][keep])

    return {
        'boxes':  torch.cat(final_boxes),
        'scores': torch.cat(final_scores),
        'labels': torch.cat(final_labels),
    }

print("TTA helper ready")

TTA helper ready


#  Predict & export

In [10]:
def predict_and_export(model, val_dir, save_csv_path,
                       save_img_dir=VIZ_DIR, use_tta=True):
    os.makedirs(save_img_dir, exist_ok=True)
    model.eval()
    rows       = []
    val_images = sorted([f for f in os.listdir(val_dir)
                         if f.lower().endswith('.jpg')])
    val_tf = get_val_transforms()

    for img_file in tqdm(val_images, desc="Predicting"):
        orig_img     = np.array(Image.open(
            os.path.join(val_dir, img_file)).convert('RGB'))
        orig_h, orig_w = orig_img.shape[:2]

        aug          = val_tf(image=orig_img, bboxes=[], class_labels=[])
        image_tensor = aug['image'].unsqueeze(0).to(DEVICE)

        if use_tta:
            output = tta_predict(model, image_tensor)
        else:
            with torch.no_grad():
                output = model(image_tensor)[0]

        # Scale boxes back to original image size
        inp_h, inp_w = image_tensor.shape[-2:]
        scale_x = orig_w / inp_w
        scale_y = orig_h / inp_h

        boxes  = output['boxes'].cpu().numpy()
        scores = output['scores'].cpu().numpy()
        labels = output['labels'].cpu().numpy()

        # Rescale
        if len(boxes):
            boxes[:, [0,2]] *= scale_x
            boxes[:, [1,3]] *= scale_y

        cf_list, xmin_list, ymin_list = [], [], []
        xmax_list, ymax_list, lbl_list = [], [], []

        fig, ax = plt.subplots(1, figsize=(5,5))
        ax.imshow(orig_img)
        ax.axis('off')

        for box, score, label in zip(boxes, scores, labels):
            xmin, ymin, xmax, ymax = map(int, box.tolist())
            lname = REV_LABEL_MAP.get(int(label), 'unknown')
            cf_list.append(f"{score:.4f}")
            xmin_list.append(str(xmin)); ymin_list.append(str(ymin))
            xmax_list.append(str(xmax)); ymax_list.append(str(ymax))
            lbl_list.append(lname)
            rect = patches.Rectangle(
                (xmin,ymin), xmax-xmin, ymax-ymin,
                linewidth=1.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, max(ymin-4,0), f"{lname} {score:.2f}",
                    color='red', fontsize=7, weight='bold',
                    bbox=dict(facecolor='white', alpha=0.5,
                              pad=1, edgecolor='none'))

        fig.tight_layout(pad=0)
        fig.savefig(os.path.join(save_img_dir, img_file),
                    bbox_inches='tight', dpi=100)
        plt.close(fig)

        if cf_list:
            best_idx = int(np.argmax([float(c) for c in cf_list]))
            rows.append({'ID': img_file, 'label': lbl_list[best_idx],
                         'cf':   ' '.join(cf_list),
                         'xmin': ' '.join(xmin_list),
                         'ymin': ' '.join(ymin_list),
                         'xmax': ' '.join(xmax_list),
                         'ymax': ' '.join(ymax_list)})
        else:
            rows.append({'ID': img_file, 'label': 'none',
                         'cf':'0','xmin':'0','ymin':'0',
                         'xmax':'0','ymax':'0'})

    df = pd.DataFrame(rows)
    df.fillna('none', inplace=True)
    df.to_csv(save_csv_path, index=False)
    print(f"\n✓ submission.csv  → {save_csv_path}")
    print(f"✓ viz images      → {save_img_dir}/")
    return df

# Run everything

In [ ]:
# Build loaders
train_dataset = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT,
                              transforms=get_train_transforms())
sampler       = get_balanced_sampler(train_dataset)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           sampler=sampler, collate_fn=collate_fn,
                           num_workers=2, pin_memory=True)

# Small val split from training set for mAP tracking
val_size     = min(200, int(0.1 * len(train_dataset)))
val_indices  = list(range(len(train_dataset) - val_size, len(train_dataset)))
val_dataset  = torch.utils.data.Subset(
    NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT,
                  transforms=get_val_transforms()),
    val_indices)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

if not INFER_ONLY:
    model = train_model(model, train_loader, val_loader, epochs=EPOCHS)
else:
    wp    = EMA_PATH if os.path.exists(EMA_PATH) else CHECKPOINT_PATH
    state = torch.load(wp, map_location=DEVICE)
    model.load_state_dict(state['model'] if 'model' in state else state)
    print(f"Loaded weights from {wp}")

submission_df = predict_and_export(
    model, VAL_IMG_ROOT, SUBMISSION_PATH,
    save_img_dir=VIZ_DIR, use_tta=USE_TTA)

submission_df.head(10)

/tmp/ipykernel_58/2067940155.py:4: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_58/2067940155.py:17: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.4),
/tmp/ipykernel_58/2067940155.py:23: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=6, max_height=32,
/tmp/ipykernel_58/2067940155.py:40: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
Epoch   1/50:   0%|          | 0/360 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist

# View results

In [ ]:
# Label distribution in predictions
print(submission_df['label'].value_counts())

# Show sample predictions
imgs = sorted(os.listdir(VIZ_DIR))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, fname in zip(axes.flatten(), imgs):
    ax.imshow(Image.open(os.path.join(VIZ_DIR, fname)))
    ax.set_title(fname.split('.')[0], fontsize=7)
    ax.axis('off')
for ax in axes.flatten()[len(imgs):]:
    ax.axis('off')
plt.suptitle('Sample predictions', fontsize=13)
plt.tight_layout()
plt.show()